In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_mpt import StockMPT, LinearModel, NaiveModel
from dataloader_builder_mpt import build_dataloaders
from setup import StockMPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor
from model_training_mpt import model_setup, train_model_cuda

from model_training_mpt import train_model_cuda, evaluate_model, evaluate_best_model, precision_recall_curve
from model_analysis_mpt import test_model, print_loss_analysis, process_losses, format_num

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

preprocessed_data/data_1min_2021_2026_2
Building DataLoaders...
Train dataset samples: 82,406
Train loader batches:  643
Batch size:            128


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockMPT, stockMPT_params, opt1, sca1, sch1 = model_setup(StockMPT, StockMPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

3415808
4608


NaiveModel()

In [5]:
model_train_losses, model_val_losses = train_model_cuda(stockMPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:

Learning Rate: 4.00e-04



|█         | 10.0% (03:01) Evaluating model on validation data... (196/197) [1483/14830]:                 

Epoch 1:
Training Loss:
   (CE)   0.8974185585975647
   (ACC)  0.5749486088752747
   (PREC) tensor([0.4558, 0.6179, 0.4364], device='cuda:0')
   (REC)  tensor([0.2368, 0.9045, 0.2207], device='cuda:0')
   (F1)   tensor([0.3117, 0.7342, 0.2932], device='cuda:0')
Validation Loss:
   (CE)   0.8956601619720459
   (ACC)  0.5742517709732056
   (PREC) tensor([0.4581, 0.6159, 0.4344], device='cuda:0')
   (REC)  tensor([0.2381, 0.9121, 0.2099], device='cuda:0')
   (F1)   tensor([0.3133, 0.7353, 0.2831], device='cuda:0')

Best Validation: 0.8956601619720459
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|██        | 20.0% (05:59) Evaluating model on validation data... (196/197) [2966/14830]: 

...

|████████  | 80.1% (01:35) Training LinearModel-v15-111-2-183... [11872/14830]:            

Epoch 8:
Training Loss:
   (CE)   0.9557600021362305
   (ACC)  0.5426633954048157
   (PREC) tensor([0.3739, 0.5943, 0.4153], device='cuda:0')
   (REC)  tensor([0.3342, 0.8834, 0.0379], device='cuda:0')
   (F1)   tensor([0.3529, 0.7105, 0.0694], device='cuda:0')
Validation Loss:
   (CE)   0.9516646265983582
   (ACC)  0.5479229688644409
   (PREC) tensor([0.3912, 0.5928, 0.4054], device='cuda:0')
   (REC)  tensor([0.3096, 0.9029, 0.0519], device='cuda:0')
   (F1)   tensor([0.3456, 0.7157, 0.0920], device='cuda:0')

Best Validation: 0.9515357613563538
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



|█████████ | 90.1% (01:49) Training LinearModel-v15-111-2-183... [13356/14830]:            

Epoch 9:
Training Loss:
   (CE)   0.9541861414909363
   (ACC)  0.5442495942115784
   (PREC) tensor([0.3844, 0.5890, 0.3985], device='cuda:0')
   (REC)  tensor([0.3073, 0.8959, 0.0447], device='cuda:0')
   (F1)   tensor([0.3415, 0.7108, 0.0804], device='cuda:0')
Validation Loss:
   (CE)   0.9506291151046753
   (ACC)  0.548825740814209
   (PREC) tensor([0.4026, 0.5879, 0.3958], device='cuda:0')
   (REC)  tensor([0.3003, 0.9125, 0.0451], device='cuda:0')
   (F1)   tensor([0.3440, 0.7151, 0.0809], device='cuda:0')

Best Validation: 0.9506291151046753
----------------------------------------------------------------------------------------------------

Learning Rate: 2.00e-04



Epoch 10:
Training Loss:
   (CE)   0.9541898965835571
   (ACC)  0.5442173480987549
   (PREC) tensor([0.3842, 0.5890, 0.3999], device='cuda:0')
   (REC)  tensor([0.3083, 0.8959, 0.0435], device='cuda:0')
   (F1)   tensor([0.3421, 0.7107, 0.0784], device='cuda:0')
Validation Loss:
   (CE)   0.9506527781486511
   (ACC)  0.5488036870956421
   (PREC) tensor([0.4025, 0.5879, 0.3966], device='cuda:0')
   (REC)  tensor([0.3008, 0.9125, 0.0444], device='cuda:0')
   (F1)   tensor([0.3443, 0.7151, 0.0798], device='cuda:0')

Best Validation: 0.9506291151046753
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [6]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
mpt_losses = evaluate_best_model(stockMPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
mpt_test_losses = test_model(dls["test"], stockMPT, device, eval_bs, analysis_pbar)


|███▎      | 32.6% (00:08) Evaluating model on training data... (18/643) [859/2631]:                      

[] []


|██████▍   | 64.3% (00:16) Evaluating model on training data... (11/643) [1692/2631]:    

[] []


|██████████| 100.0% (01:11) Evaluating model on testing data... (36/37) [2631/2631]:     

In [7]:
for key, features in [("CE", StockMPT_cfg["target_features"]),
                      ("ACC", StockMPT_cfg["target_features"]),
                      ("PREC", StockMPT_cfg["target_features"]),
                      ("REC", StockMPT_cfg["target_features"]),
                      ("F1", StockMPT_cfg["target_features"])]:
    print_loss_analysis(
        process_losses(mpt_losses + mpt_test_losses +
                       linear_losses + linear_test_losses +
                       naive_losses + naive_test_losses, key),
        [stockMPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
        [format_num(stockMPT_params), format_num(linearModel_params), "0"],
        features, key
    )


--------------------------------------------------------------------------------------------------------------

CE

--------------------------------------------------------------------------------------------------------------

StockMPT-v15-111-2-183: 3.4M
    Training:       0.8697
    Validation:     0.8703
    Testing:        0.8419
    
LinearModel-v15-111-2-183: 4.6K
    Training:       0.9542
    Validation:     0.9506
    Testing:        0.9252
    
NaiveModel-B1_183: 0
    Training:       1.0245
    Validation:     1.0243
    Testing:        0.9981
    

--------------------------------------------------------------------------------------------------------------



--------------------------------------------------------------------------------------------------------------

ACC

--------------------------------------------------------------------------------------------------------------

StockMPT-v15-111-2-183: 3.4M
    Training:       0.5909
    Validation:     0.5875
    

In [8]:
precision_recall_curve(dls["val"], stockMPT, device, cls=2)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=1)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=0)

0.10 | PREC 0.2965 | REC 0.9442 | N 7641959
0.15 | PREC 0.3239 | REC 0.8804 | N 6523164
0.20 | PREC 0.3498 | REC 0.7981 | N 5474939
0.25 | PREC 0.3747 | REC 0.7002 | N 4484306
0.30 | PREC 0.3987 | REC 0.5844 | N 3517407
0.35 | PREC 0.4238 | REC 0.4503 | N 2550095
0.40 | PREC 0.4511 | REC 0.2967 | N 1578139
0.45 | PREC 0.4882 | REC 0.1470 | N 722666
0.50 | PREC 0.5473 | REC 0.0520 | N 227947
0.55 | PREC 0.5968 | REC 0.0206 | N 82756
0.60 | PREC 0.6394 | REC 0.0092 | N 34384
0.65 | PREC 0.6798 | REC 0.0039 | N 13790
0.70 | PREC 0.7254 | REC 0.0014 | N 4793
0.75 | PREC 0.7625 | REC 0.0004 | N 1284
0.80 | PREC 0.7990 | REC 0.0001 | N 204
0.85 | PREC 0.8333 | REC 0.0000 | N 6
0.90 | PREC 0.0000 | REC 0.0000 | N 0
0.95 | PREC 0.0000 | REC 0.0000 | N 0
--------------


|██████████| 100.0% (01:26) Evaluating model on testing data... (36/37) [2631/2631]: 

0.10 | PREC 0.5267 | REC 0.9947 | N 9333984
0.15 | PREC 0.5471 | REC 0.9839 | N 8887034
0.20 | PREC 0.5715 | REC 0.9660 | N 8353091
0.25 | PREC 0.5968 | REC 0.9425 | N 7803512
0.30 | PREC 0.6218 | REC 0.9143 | N 7266444
0.35 | PREC 0.6464 | REC 0.8819 | N 6742448
0.40 | PREC 0.6705 | REC 0.8455 | N 6231420
0.45 | PREC 0.6945 | REC 0.8047 | N 5726206
0.50 | PREC 0.7184 | REC 0.7591 | N 5222235
0.55 | PREC 0.7423 | REC 0.7083 | N 4715673
0.60 | PREC 0.7669 | REC 0.6508 | N 4194037
0.65 | PREC 0.7917 | REC 0.5866 | N 3661564
0.70 | PREC 0.8171 | REC 0.5146 | N 3112335
0.75 | PREC 0.8432 | REC 0.4342 | N 2545065
0.80 | PREC 0.8705 | REC 0.3438 | N 1951744
0.85 | PREC 0.9000 | REC 0.2463 | N 1352507
0.90 | PREC 0.9301 | REC 0.1484 | N 788481
0.95 | PREC 0.9593 | REC 0.0555 | N 285833
--------------
0.10 | PREC 0.3008 | REC 0.9399 | N 7394910
0.15 | PREC 0.3317 | REC 0.8712 | N 6215849
0.20 | PREC 0.3601 | REC 0.7870 | N 5173184
0.25 | PREC 0.3866 | REC 0.6888 | N 4217000
0.30 | PREC 0.4125 

In [9]:
dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [10]:
test_losses = test_model(dls["test"], stockMPT, device, eval_bs)
print(test_losses)

KeyError: 'test'